In [ ]:
///- --package ..
:< sm'_operators
:< engine

#### index (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:= year_of (e : page.entry) : string = sm'.slice 0i32 3i32 e.date

:= entry_row (root : string) (e : page.entry) : string =
    "<li class=\"post\"><time>" ++# html.escape e.date ++# "</time>"
    ++# "<span class=\"kind\">" ++# page.kind_label e.kind ++# "</span>"
    ++# "<a href=\"" ++# root ++# html.escape e.slug ++# "/\">" ++# html.escape e.title ++# "</a>"
    ++# "<p>" ++# html.escape e.blurb ++# "</p></li>"

:= ^ rows (root : string) (current : string) (entries : list page.entry) : string =
    ?= entries ?|
    | [] => ? current = "" ?> "" |: "</ul>"
    | e :: rest =>
        := year = year_of e
        ? year = current
        ?> entry_row root e ++# rows root current rest
        |:
            := close = ? current = "" ?> "" |: "</ul>"
            close ++# "<h2 class=\"year\">" ++# year ++# "</h2><ul>" ++# entry_row root e ++# rows root year rest

:= featured (root : string) (entries : list page.entry) : string =
    ?= entries ?|
    | [] => ""
    | e :: _ => "<section class=\"featured\"><span class=\"badge\">Latest</span>" ++# entry_row root e ++# "</section>"

:= ^ only (kind : page.kind) (entries : list page.entry) : list page.entry =
    ?= entries ?|
    | [] => []
    | e :: rest => ? e.kind = kind ?> e :: only kind rest |: only kind rest

:= render_index (root : string) (entries : list page.entry) : string =
    "<nav class=\"tabs\"><a href=\"" ++# root ++# "\">Everything</a>"
    ++# "<a href=\"" ++# root ++# "articles/\">Articles</a>"
    ++# "<a href=\"" ++# root ++# "releases/\">Releases</a></nav>"
    ++# featured root entries
    ++# rows root "" entries

#### the_year_comes_from_the_date (test)

In [1]:
///- --test --symbol
///> zig

year_of (page.sample ()) |> _assert_eq "2026"

__assert_eq { expected = 2026 }

#### the_index_features_the_latest_post (test)

In [2]:
///- --test --symbol
///> zig

render_index "/" [ page.sample () ] |> sm'.contains "<span class=\"badge\">Latest</span>" |> _assert_eq !+

__assert_eq { expected = true }

#### posts_are_grouped_under_their_year (test)

In [3]:
///- --test --symbol
///> zig

render_index "/" [ page.sample () ] |> sm'.contains "<h2 class=\"year\">2026</h2><ul>" |> _assert_eq !+

__assert_eq { expected = true }

#### a_tab_keeps_only_its_kind (test)

In [4]:
///- --test --symbol
///> zig

:= article = page.sample ()
:= release = { article ?| slug = "v3"; kind = page.Release }
[ article; release ] |> only page.Release |> listm.length |> _assert_eq 1i32

__assert_eq { expected = 1 }

#### links_start_at_the_site_root (test)

In [5]:
///- --test --symbol
///> zig

render_index "/blog/" [ page.sample () ] |> sm'.contains "<a href=\"/blog/sample/\">" |> _assert_eq !+

__assert_eq { expected = true }